# RT-DETR-L 알약 객체 탐지

RT-DETR-L을 학습하고 Kaggle 제출 형식의 객체 탐지 결과를 생성합니다.

셀 출력과 epoch별 학습 로그는 저장하지 않은 제출용 정리본입니다. 위에서 아래 순서로 실행합니다.

## 1. 실행 환경

필요한 패키지를 설치하고 GPU 사용 가능 여부를 확인합니다.

In [ ]:
!pip install -q -U ultralytics pyyaml


In [ ]:
from pathlib import Path
from collections import defaultdict
import gc
import json
import shutil

import pandas as pd
import torch
import yaml

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from ultralytics import RTDETR


## 2. 경로와 실험 설정

Kaggle Input에서 학습 데이터와 842장의 테스트 이미지를 자동으로 탐색합니다. 모델 설정은 이 셀에서만 변경합니다.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
EXPECTED_TEST_IMAGES = 842
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_training_dataset_root(input_root):
    """Find the single dataset containing COCO annotations and split images."""
    candidates = []
    for train_json in input_root.rglob("train.json"):
        root = train_json.parent
        required = [
            root / "val.json",
            root / "images" / "train",
            root / "images" / "val",
        ]
        if all(path.exists() for path in required):
            candidates.append(root)

    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one training dataset root, found: {candidates}")
    return candidates[0]


def find_test_image_dir(input_root, expected_count=EXPECTED_TEST_IMAGES):
    """Find the Kaggle test directory by image count and numeric file names."""
    candidates = []
    for directory in input_root.rglob("*"):
        if not directory.is_dir() or directory.name.lower() not in {"test", "test_images", "images"}:
            continue
        images = [
            path for path in directory.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES
        ]
        if len(images) == expected_count and all(path.stem.isdigit() for path in images):
            candidates.append(directory)

    candidates = sorted(set(candidates))
    if not candidates:
        print(f"WARNING: no {expected_count}-image test directory found; submission will be skipped")
        return None
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one test image directory, found: {candidates}")
    return candidates[0]


DATASET_ROOT = find_training_dataset_root(INPUT_ROOT)
IMAGE_ROOT = DATASET_ROOT / "images"
TEST_DIR = find_test_image_dir(INPUT_ROOT)
CONVERTED_ROOT = WORK_ROOT / "pill_detection_dataset"

print("DATASET_ROOT:", DATASET_ROOT)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("TEST_DIR:", TEST_DIR)

TRAIN_MODEL = True
IMGSZ = 960
EPOCHS = 25
BATCH = 8
CONFIDENCES = [0.05, 0.15, 0.25]
DEFAULT_CONFIDENCE = 0.25
IOU = 0.70
MAX_DET = 10
SEED = 42
RUN_NAME = "rtdetr_l_960"
RUN_DIR = WORK_ROOT / "rtdetr_runs" / RUN_NAME


## 3. 데이터 변환

COCO의 비연속 category ID를 0부터 시작하는 학습용 class index로 변환합니다. 원래 category ID는 `class_map.json`에 보존하여 제출 파일 생성 시 복원합니다.

In [ ]:
def convert_coco_split(split, category_to_index):
    """Convert one COCO split to normalized Ultralytics detection labels."""
    json_path = DATASET_ROOT / f"{split}.json"
    source_image_dir = IMAGE_ROOT / split
    output_image_dir = CONVERTED_ROOT / "images" / split
    output_label_dir = CONVERTED_ROOT / "labels" / split
    output_image_dir.mkdir(parents=True, exist_ok=True)
    output_label_dir.mkdir(parents=True, exist_ok=True)

    with json_path.open("r", encoding="utf-8") as file:
        coco = json.load(file)

    images = {int(image["id"]): image for image in coco["images"]}
    labels_by_image = defaultdict(list)

    for annotation in coco["annotations"]:
        image = images.get(int(annotation["image_id"]))
        if image is None:
            continue

        x, y, width, height = map(float, annotation["bbox"])
        if width <= 0 or height <= 0:
            continue

        image_width = float(image["width"])
        image_height = float(image["height"])
        class_index = category_to_index[int(annotation["category_id"])]
        normalized_box = [
            (x + width / 2) / image_width,
            (y + height / 2) / image_height,
            width / image_width,
            height / image_height,
        ]
        if not all(0 <= value <= 1 for value in normalized_box):
            continue

        labels_by_image[int(annotation["image_id"])].append(
            f"{class_index} " + " ".join(f"{value:.6f}" for value in normalized_box)
        )

    missing_images = []
    for image_id, image in images.items():
        file_name = Path(image["file_name"]).name
        source_image = source_image_dir / file_name
        output_image = output_image_dir / file_name
        output_label = output_label_dir / f"{Path(file_name).stem}.txt"

        if not source_image.exists():
            missing_images.append(source_image)
            continue
        if not output_image.exists():
            shutil.copy2(source_image, output_image)
        output_label.write_text("\n".join(labels_by_image[image_id]), encoding="utf-8")

    if missing_images:
        raise FileNotFoundError(
            f"{split}: {len(missing_images)} images are missing; first={missing_images[0]}"
        )
    return len(images), sum(map(len, labels_by_image.values()))


with (DATASET_ROOT / "train.json").open("r", encoding="utf-8") as file:
    train_coco = json.load(file)

categories = sorted(train_coco["categories"], key=lambda item: int(item["id"]))
category_to_index = {
    int(category["id"]): index for index, category in enumerate(categories)
}
class_map = {
    index: int(category["id"]) for index, category in enumerate(categories)
}
class_names = [category["name"] for category in categories]

train_count = convert_coco_split("train", category_to_index)
val_count = convert_coco_split("val", category_to_index)

data_yaml = CONVERTED_ROOT / "data.yaml"
with data_yaml.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        {
            "path": str(CONVERTED_ROOT),
            "train": "images/train",
            "val": "images/val",
            "nc": len(class_names),
            "names": class_names,
        },
        file,
        allow_unicode=True,
        sort_keys=False,
    )

(CONVERTED_ROOT / "class_map.json").write_text(
    json.dumps(class_map, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("train images/labels:", train_count)
print("val images/labels:", val_count)
print("classes:", len(class_map))
print("data yaml:", data_yaml)


## 4. 데이터 검증

학습 전에 이미지와 라벨의 쌍, 클래스 범위, 정규화 좌표를 검사합니다. 테스트 데이터가 연결되지 않아도 학습은 진행할 수 있습니다.

In [ ]:
def verify_dataset_split(split):
    """Validate image-label pairs, class indices, and normalized box values."""
    image_dir = CONVERTED_ROOT / "images" / split
    label_dir = CONVERTED_ROOT / "labels" / split
    images = {
        path.stem for path in image_dir.iterdir()
        if path.suffix.lower() in IMAGE_SUFFIXES
    }
    labels = {path.stem for path in label_dir.glob("*.txt")}
    if images != labels:
        raise RuntimeError(f"{split}: image/label mismatch")

    bad_lines = []
    for label_path in label_dir.glob("*.txt"):
        for line_number, line in enumerate(
            label_path.read_text(encoding="utf-8").splitlines(), start=1
        ):
            parts = line.split()
            if len(parts) != 5:
                bad_lines.append((label_path.name, line_number, line))
                continue
            class_index = int(parts[0])
            coordinates = [float(value) for value in parts[1:]]
            if class_index not in class_map or not all(0 <= value <= 1 for value in coordinates):
                bad_lines.append((label_path.name, line_number, line))

    if bad_lines:
        raise RuntimeError(f"{split}: invalid label lines; examples={bad_lines[:5]}")
    print(split, "verified:", len(images), "images")


verify_dataset_split("train")
verify_dataset_split("val")

if TEST_DIR is None:
    test_images = []
else:
    test_images = sorted(
        [path for path in TEST_DIR.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES],
        key=lambda path: int(path.stem),
    )
    if len(test_images) != EXPECTED_TEST_IMAGES:
        raise RuntimeError(f"Unexpected test image count: {len(test_images)}")

print("test images:", len(test_images) if test_images else "not attached")


## 5. RT-DETR-L 학습

사전학습 가중치로 학습을 시작하고 검증 성능이 가장 높은 `best.pt`를 Kaggle Output 최상단에 보존합니다.

In [ ]:
if TRAIN_MODEL:
    model = RTDETR("rtdetr-l.pt")
    model.train(
        data=str(data_yaml),
        imgsz=IMGSZ,
        epochs=EPOCHS,
        batch=BATCH,
        optimizer="AdamW",
        lr0=0.0001,
        weight_decay=0.0001,
        patience=20,
        seed=SEED,
        project=str(WORK_ROOT / "rtdetr_runs"),
        name=RUN_NAME,
        exist_ok=True,
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()

best_model = RUN_DIR / "weights" / "best.pt"
if not best_model.exists():
    raise FileNotFoundError(best_model)

saved_model = WORK_ROOT / "rtdetr_l_960_best.pt"
shutil.copy2(best_model, saved_model)
print("best model:", saved_model)


## 6. 제출 후보 생성

GPU 메모리 사용량을 제한하기 위해 한 장씩 스트리밍 추론합니다. 예측 class index는 원래 COCO category ID로 되돌리고 confidence별 CSV를 생성합니다.

In [ ]:
SUBMISSION_COLUMNS = [
    "annotation_id",
    "image_id",
    "category_id",
    "bbox_x",
    "bbox_y",
    "bbox_w",
    "bbox_h",
    "score",
]


def predict_to_dataframe(model_path, imgsz, minimum_confidence):
    """Run streaming inference and return valid Kaggle-format detections."""
    model_path = Path(model_path)
    if not model_path.exists():
        raise FileNotFoundError(model_path)
    if TEST_DIR is None:
        raise RuntimeError("Kaggle test images are not attached")

    model = RTDETR(str(model_path))
    results = model.predict(
        source=str(TEST_DIR),
        imgsz=imgsz,
        conf=minimum_confidence,
        iou=IOU,
        max_det=MAX_DET,
        batch=1,
        stream=True,
        verbose=False,
    )

    rows = []
    for result in results:
        image_id = int(Path(result.path).stem)
        if result.boxes is None:
            continue
        image_height, image_width = result.orig_shape
        for box in result.boxes:
            class_index = int(box.cls.item())
            if class_index not in class_map:
                raise KeyError(f"Missing class mapping for index {class_index}")

            x1, y1, x2, y2 = box.xyxy[0].cpu().tolist()
            x1 = max(0.0, min(x1, float(image_width)))
            y1 = max(0.0, min(y1, float(image_height)))
            x2 = max(0.0, min(x2, float(image_width)))
            y2 = max(0.0, min(y2, float(image_height)))
            width = x2 - x1
            height = y2 - y1
            if width <= 0 or height <= 0:
                continue

            rows.append({
                "image_id": image_id,
                "category_id": class_map[class_index],
                "bbox_x": x1,
                "bbox_y": y1,
                "bbox_w": width,
                "bbox_h": height,
                "score": float(box.conf.item()),
            })

    del results, model
    gc.collect()
    torch.cuda.empty_cache()
    return pd.DataFrame(rows, columns=SUBMISSION_COLUMNS[1:])


def save_confidence_candidates(raw_predictions, model_name, confidences):
    """Save one validated submission file for each confidence threshold."""
    output_paths = {}
    for confidence in confidences:
        filtered = raw_predictions[raw_predictions["score"] >= confidence].copy()
        filtered = filtered.sort_values(
            ["image_id", "score"], ascending=[True, False]
        ).reset_index(drop=True)
        filtered.insert(0, "annotation_id", range(1, len(filtered) + 1))
        filtered = filtered[SUBMISSION_COLUMNS]

        confidence_name = f"{int(confidence * 1000):03d}"
        output_path = WORK_ROOT / (
            f"submission_{model_name}_conf{confidence_name}_max{MAX_DET}.csv"
        )
        filtered.to_csv(output_path, index=False)
        output_paths[confidence] = output_path
        print(
            output_path.name,
            filtered.shape,
            "images:",
            filtered["image_id"].nunique(),
        )
    return output_paths


In [ ]:
if TEST_DIR is not None:
    predictions = predict_to_dataframe(best_model, IMGSZ, min(CONFIDENCES))
    output_paths = save_confidence_candidates(
        predictions,
        "rtdetr_l_960",
        CONFIDENCES,
    )
    shutil.copy2(output_paths[DEFAULT_CONFIDENCE], WORK_ROOT / "submission_rtdetr.csv")
else:
    print("Test images are not attached; submission generation skipped")


## 7. 산출물 확인

가중치와 제출 CSV의 존재 여부, 행 수, 이미지 수, 컬럼 순서를 확인합니다.

In [ ]:
print("Model artifacts:")
for path in sorted(WORK_ROOT.glob("*.pt")):
    print(f"- {path.name}: {path.stat().st_size / 1024**2:.1f} MB")

print("\nSubmission files:")
for path in sorted(WORK_ROOT.glob("submission*.csv")):
    dataframe = pd.read_csv(path)
    columns_ok = list(dataframe.columns) == SUBMISSION_COLUMNS
    print(
        f"- {path.name}: rows={len(dataframe)}, "
        f"images={dataframe.image_id.nunique()}, columns_ok={columns_ok}"
    )
